# 3D Fluorescence Nuclei Segmentation

This notebook is a **generic template** for 3D fluorescence image segmentation and quantification of nuclei and cell-associated markers.

**Before running the workflow:**
- Replace the example file path, ROI, experiment name, and channel definitions in **Section 3** with your own data.
- Make sure the channel names in `stain_dict` match the metadata printed after loading your file.
- Start with the default settings and tune only the parameters that are necessary for your dataset.

---

# 1. Install Required Packages (run only once)
If you see errors about missing packages, run this cell.

In [ ]:
from helpers.notebook_setup_helpers import install_required_packages

install_required_packages()


# 2. Import Required Libraries
This cell imports all necessary libraries for image processing and visualization.
If you see an ImportError, make sure to install the missing package using pip (see cell 1).

In [ ]:
# Check OpenCV version
import cv2
cv2.__version__

In [ ]:
from helpers.notebook_setup_helpers import load_common_imports
from tqdm.auto import tqdm
from IPython.display import clear_output


globals().update(load_common_imports(profile="nuclei", enable_napari_interactive=True))


def step_progress(iterable, desc, **kwargs):
    # Keep a single visible progress bar by clearing previous loop output first.
    clear_output(wait=True)
    kwargs.pop("leave", None)
    kwargs.pop("position", None)
    kwargs.pop("dynamic_ncols", None)
    return tqdm(iterable, desc=desc, leave=False, position=0, dynamic_ncols=True, **kwargs)


#### Helper Functions



The reusable helper functions for this notebook are stored in `helpers/notebook_helpers.py`.

This keeps the notebook much lighter and makes the workflow sections easier to follow.

If you update the helper file, rerun the next code cell to reload the helpers.


In [ ]:
from importlib import reload

import helpers.notebook_helpers as nuclei_helpers

reload(nuclei_helpers)
from helpers.notebook_helpers import *

print("Helper functions loaded from helpers/notebook_helpers.py")

# 3. Inputs and Setup

Replace the example values in the following cells with information from **your own dataset** before running the analysis.

In this section, you will:
- Provide the image file path (`.nd2`, `.tif`, or similar)
- Define your marker/channel mapping
- Optionally crop a region of interest (ROI)
- Set approximate object sizes and processing options

> The `NUCLEI` entry in `stain_dict` is required, and the channel names should match the file metadata exactly.

### File input and ROI

In [ ]:
# --- User input: file path and optional ROI ---
# Replace `input_file` with your own microscopy file.
# Supported examples include `.nd2` and multi-channel `.tif/.tiff` files.
input_file = 'path/to/your_image_file.nd2'

# Set to True to load a selected ROI from large datasets.
# Set to False to load the full stack directly.
big_image = True

# ROI format: [x0, x1, y0, y1, z0, z1]
# Use 0 for any limit you want to keep unchanged.
# Example: ROI = [100, 900, 150, 1200, 0, 40]
ROI = [0, 0, 0, 0, 0, 0]

In [ ]:
meta = AICSImage(input_file)
if big_image:
    x0, x1, y0, y1, z0, z1 = ROI
    if x1==0:
        x1 = meta.shape[4]
    if y1==0:
        y1 = meta.shape[3]
    if z1==0:
        z1 = meta.shape[2]
    # Get lazy dask array in a known order, e.g. "TCZYX"
    lazy = meta.get_image_dask_data("ZYXC")
    sub = lazy[z0:z1, y0:y1, x0:x1, :]
    # Actually load this subset into memory
    img = sub.compute()
    ROI = [0, 0, 0, 0, 0, 0]
else:
    img = meta.get_image_data("XYZ", T=0) 

print(img.shape)

In [ ]:
# Read voxel size metadata from the file.
# The channel names printed below should be used when editing `stain_dict`.
r_X = meta.physical_pixel_sizes.X  # um/px
r_Y = meta.physical_pixel_sizes.Y  # um/px
r_Z = meta.physical_pixel_sizes.Z  # um/px
print([r_X, r_Y, r_Z])

if big_image == False:
    imdata = meta.get_image_data()
    imtype = imdata.dtype
    bdepth = imtype.itemsize * 8
    print(imtype)

with ND2Reader(input_file) as nd2:
    print("Date:", nd2.metadata.get("date"))
    print("Channels:", nd2.metadata.get("channels"))

### Sample and channel metadata

In [ ]:
# --- User input: biological and channel metadata ---
# Replace the example values below with those appropriate for your sample.

# Approximate object diameters used during segmentation.
nuclei_diameter = 10  # um, typical nucleus diameter in your sample
cell_diameter = 30  # um, typical whole-cell diameter if cytoplasm is estimated

# Relative expansion factors used when generating cytoplasm and PCM masks.
cyto_factor = 3.0
PCM_factor = 4.0

# Define the channels present in your dataset.
# Format:
# 'CONDITION_NAME': ['Marker label', 'channel name from file metadata', 'display color']
# Notes:
# - Keep the `NUCLEI` condition for the nuclear stain.
# - Replace the example channel names below so they match your file exactly.
stain_dict = {
    'NUCLEI': ['MARKER_NUCLEI', 'CHANNEL_NUCLEI', 'COLOR_NUCLEI'],
    'MARKER_1': ['MARKER_1', 'CHANNEL_1', 'COLOR_1'],
    'MARKER_2': ['MARKER_2', 'CHANNEL_2', 'COLOR_2'],
    'MARKER_3': ['MARKER_3', 'CHANNEL_3', 'COLOR_3'],
}

# Optional: list marker labels that should contribute to cytoplasm expansion
# if you do not have an explicit `CYTOPLASM` channel.
cyto_markers = ['MARKER_1', 'MARKER_2']  # replace with your own marker labels, or set to []

# Nucleus splitting by shrinking is configured in the helper module.
# Recommended start: "balanced"; use "aggressive" for clustered nuclei,
# or "conservative" if single nuclei are being over-split.
nuclei_split_config = get_nuclei_split_config(profile="balanced")

# Optional example overrides:
# nuclei_split_config["nuclei_seed_min_fraction"] = 0.05
# nuclei_split_config["nuclei_min_roundness"] = 0.40

### Image

In [ ]:
# Optional scaling factors.
# Keep these values at 1.0 unless you intentionally want to apply an extra manual scale correction.
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]
zoom_factors = [x * scale_factor for x in zoom_factors]

### Setup

In [ ]:
# --- User input: experiment-level processing options ---
# This prefix is used when saving setup files and output tables.
name_setup = 'your_experiment_name'

# Reuse previously saved napari contrast/gamma settings if the CSV file exists.
use_setup = True

# Set to True to use the StarDist model instead of watershed-based nuclei segmentation.
trig_stardist = False

# Set to True to compute multi-marker combinations in the quantification summary.
multilabel = True

# 4. Information

This section calculates derived image parameters (cell and nuclei dimensions, volumes), builds the staining table from your inputs, and opens an interactive Napari viewer for contrast and gamma setup.

In [ ]:
nuclei_radius=nuclei_diameter*0.5 #um
cell_radius=cell_diameter*0.5 #um

nuclei_volume=np.ceil(4.0*((nuclei_radius)**3.0)*np.pi/3.0) #um^3
cell_volume=np.ceil(4.0*((cell_radius)**3.0)*np.pi/3.0) #um^3

x0, x1, y0, y1, z0, z1 = ROI

if x1==0:
    x1 = img.shape[0]
if y1==0:
    y1 = img.shape[1]
if z1==0:
    z1 = img.shape[2]

if big_image:
    im_original = img.astype('float32')
    im_original_ROI = im_original.copy()
else:
    im_original = meta.get_image_data("ZYXC", S=0, T=0).astype('float32')
    im_original_ROI = im_original[z0:z1,y0:y1,x0:x1,:]

im_final_stack={'Original image': im_original_ROI}

In [ ]:
if big_image:
    # Ensure `Original image` has axes Z,Y,X,C and squeeze singleton time axis
    orig = im_final_stack['Original image']
    try:
        shape = orig.shape
    except Exception:
        shape = None
    print('Original image shape before fix:', shape)
    if shape is not None:
        if len(shape) == 5:
            # common order Z,Y,X,C,T -> drop T if singleton
            if shape[4] == 1:
                orig = orig[..., 0]
        if len(orig.shape) == 4:
            # ensure channels are last: if last axis looks large (likely X), detect small axis as channel
            if orig.shape[-1] > 50:
                chan_axis = next((i for i, s in enumerate(orig.shape) if s < 50), None)
                if chan_axis is not None and chan_axis != 3:
                    import numpy as np
                    orig = np.moveaxis(orig, chan_axis, -1)
    im_final_stack['Original image'] = orig
    print('Original image shape after fix:', getattr(orig, 'shape', None))

### Information about the staining

In [ ]:
# Build the staining table from `stain_dict`.
# Make sure the `Laser` strings in `stain_dict` match the channel names printed above.
stain_dict = {k.upper(): [item.upper() if isinstance(item, str) else item for item in v] for k, v in stain_dict.items()}
stain_df = pd.DataFrame.from_dict(stain_dict, orient='index', columns=['Marker', 'Laser', 'Color'])
laser_order = nd2.metadata.get("channels")

# Map each fluorophore/channel name to its order in the file metadata
order_map = {name.strip().upper(): i for i, name in enumerate(laser_order)}
stain_df['order'] = stain_df['Laser'].map(order_map)

# Sort by channel order and drop the temporary helper column
stain_df = stain_df.sort_values('order').drop(columns='order')

stain_df.index.name = 'Condition'

if 'NUCLEI' not in stain_df.index:
    print('No nuclei condition!')

In [ ]:
# Visualize each channel using napari
im_in=im_final_stack['Original image'].copy()

viewer_0 = napari.Viewer()
for c, c_name in step_progress(
    enumerate(stain_df['Marker']),
    total=len(stain_df['Marker']),
    desc='Step 04 - Visualize Channels',
    leave=False,
):
    #im_in = meta.get_image_data("ZYX", C=c, S=0, T=0).astype('float32')
    im_channel = im_in[:,:,:,c]

    # Stretch to [0, 255]
    im_8b = ((im_channel - im_channel.min()) / (im_channel.max() - im_channel.min()) * 255).clip(0, 255).astype('uint8')
    
    viewer_0.add_image(im_8b, name=f"{stain_df.index[c]} ({c_name})", 
                        colormap=stain_df['Color'][c], blending='additive')

    viewer_0.scale_bar.visible = True
    viewer_0.scale_bar.unit = 'um'

### Acquisition processing setup

In [ ]:
# Setup for acquisition and contrast/gamma settings
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    settings=settings,
    napari_module=napari,
    progress=step_progress,
)

In [ ]:
# Display stain settings DataFrame
stain_complete_df

# 5. Image Processing and Segmentation

This section normalizes the image, resamples to isotropic voxels, denoises, applies contrast and gamma adjustments, and then segments the structures of interest.

In [ ]:
# Load and normalize image data for all channels
im_final_stack['Normalized image'] = normalize_image_channels(im_final_stack['Original image'])

# Plot histogram for each channel
hist_plot(im_final_stack['Normalized image'], stain_complete_df)


In [ ]:
# Adapt resolution to isotropic
im_final_stack['Zoomed image'], r_zX, r_zY, r_zZ = resample_to_isotropic(
    im_final_stack['Normalized image'],
    zoom_factors,
    meta=meta
)

hist_plot(im_final_stack['Zoomed image'], stain_complete_df)


In [ ]:
# Noise removal using median filter
im_final_stack['Denoised image'] = apply_median_denoise(im_final_stack['Zoomed image'])
hist_plot(im_final_stack['Denoised image'], stain_complete_df)


In [ ]:
# Contrast and gamma adjustment for each channel
im_final_stack['Adjusted image'] = apply_contrast_gamma_per_channel(
    im_final_stack['Denoised image'],
    stain_complete_df
)
hist_plot(im_final_stack['Adjusted image'], stain_complete_df)


In [ ]:
# Gaussian filter for smoothing
im_final_stack['Filtered image'] = apply_gaussian_smoothing(
    im_final_stack['Adjusted image'],
    sigma=0.5
)
hist_plot(im_final_stack['Filtered image'], stain_complete_df)


In [ ]:
# Export histograms for each channel
output_path = Path(input_file).stem + '_histograms.xlsx'
export_channel_histograms(
    im_final_stack['Adjusted image'],
    stain_complete_df,
    output_path,
    progress=step_progress,
)
print(f"Saved to: {output_path}")

In [ ]:
# Histogram equalization, supporting thresholding
im_final_stack['Equalized image'] = apply_histogram_equalization_per_channel(
    im_final_stack['Filtered image'],
    num_plateaus=2,
    plateau_factor=0.7
)
hist_plot(im_final_stack['Equalized image'], stain_complete_df)


In [ ]:
# Thresholding using Otsu, Sauvola, statistical background, gain filtering
im_in = im_final_stack["Equalized image"].copy()
im_out = im_in.copy()

# Sizes
nuclei_size = int(nuclei_diameter / (np.mean([r_zX, r_zY])))
cell_size = int(cell_diameter / (np.mean([r_zX, r_zY])))

for c in step_progress(range(im_in.shape[3]), desc='Step 09 - Threshold Channels'):
    img = sitk.GetImageFromArray(im_in[:, :, :, c])

    # Stretch for Otsu
    rescaler = sitk.RescaleIntensityImageFilter()
    rescaler.SetOutputMinimum(0)
    rescaler.SetOutputMaximum(255)
    stretched = rescaler.Execute(img)


    # Otsu thresholds
    th_filter = sitk.OtsuThresholdImageFilter()
    _ = th_filter.Execute(stretched)
    otsu_value = th_filter.GetThreshold()

    _ = th_filter.Execute(img)
    otsu_value2 = th_filter.GetThreshold()

    if stain_complete_df.index[c] == "NUCLEI":
        window_size = 1 * nuclei_size #+ 1
    else:
        window_size = 4 * cell_size + 1

    # Convert to array
    arr = sitk.GetArrayFromImage(img) #.astype(np.float32)

    # Sauvola threshold map
    sauvola_value = threshold_sauvola(arr, window_size=int(window_size))

    # -------- GLOBAL statistical background, excluding zeros --------
    non_zero = arr[arr > 0]

    if non_zero.size > 0:
        hist, bins = np.histogram(non_zero, bins=256, range=(0, non_zero.max()))
        mode_bin = bins[np.argmax(hist)]
        print(mode_bin)
        bg_mask = (arr >= mode_bin - 5) & (arr <= mode_bin + 5) & (arr > 0) 
        gain_tot=6.0

        gain_ass=gain_tot*(255.0-4.0*mode_bin)/255.0
        bg_vals = arr[bg_mask]

        if bg_vals.size < 50:
            p10 = np.percentile(non_zero, 10)
            bg_vals = non_zero[non_zero <= p10]
    else:
        bg_vals = arr

    bg_mean = bg_vals.mean()
    bg_std = bg_vals.std() + 1e-6

    bg_mean_z = arr.mean()
    bg_std_z = arr.std() + 1e-6
    z = 3.0
    statistical_thr = bg_mean_z + z * bg_std_z

    # ---------------------------------------------------------------

    # 1) Soften Sauvola if it is too aggressive for large/bright cells
    # Clip Sauvola so it cannot exceed a few std above the global (zero‑including) mean
    max_sauvola = bg_mean_z + 2.0 * bg_std_z
    sauvola_clipped = np.minimum(sauvola_value, max_sauvola)

    # Final combined threshold map (slightly less Sauvola weight)
    final_thr = (
        0.60 * sauvola_clipped +
        0.25 * statistical_thr +
        0.15 * otsu_value2
    )

    # Extra improvement: intensity gain check (global, using non-zero-based bg_mean)
    gain = arr / (bg_mean + 1e-6)
    mask_gain = gain > gain_ass    # tune 2–5 depending on SNR

    # 2) Rescue pixels: strong gain but slightly under final_thr
    primary = (arr > final_thr) & mask_gain
    rescue = (gain > (gain_ass+3.0)) & (arr > statistical_thr)   # gain threshold > primary, to keep it conservative

    arrayseg = primary | rescue

    if stain_complete_df.index[c] != 'NUCLEI':
        min_size = np.ceil(0.8 * np.pi * ((nuclei_size / 2) ** 2))
    else:
        min_size= np.ceil(0.4 * np.pi * ((nuclei_size / 2) ** 2))

    # Remove small islands
    
    im_out[:, :, :, c] = remove_small_islands(arrayseg, min_size)

im_final_stack["Threshold image"] = im_out.copy()

In [ ]:
# Segmentation of nuclei using watershed or StarDist
from skimage.segmentation import relabel_sequential

if 'NUCLEI' in stain_df.index:
    im_in = im_final_stack['Threshold image'].copy()
    split_cfg = dict(nuclei_split_config)

    for c in step_progress(range(im_in.shape[3]), desc='Step 10 - Segment Nuclei'):
        if stain_complete_df.index[c] == 'NUCLEI':
            if trig_stardist:
                im_in = im_final_stack['Filtered image'].copy()
                transl = stardist3d_from_2d(
                    img_3d=im_in[:, :, :, c],
                    nucleus_radius=nuclei_diameter / 2.0,
                    voxel_size=(r_zZ, r_zY, r_zX),
                )
                im_mask = transl > 0
                im_mask = morphology.binary_erosion(
                    im_mask, footprint=np.ones((2, 2, 2))
                ).astype(im_mask.dtype)
                im_out, num = label((transl * im_mask) > 0)

            else:
                # Watershed-based segmentation with multi-scale erosion and EDT fallback
                binary_mask = im_in[:, :, :, c].astype(bool)

                im_out, debug_info = segment_nuclei_watershed(
                    binary_mask=binary_mask,
                    r_zX=r_zX,
                    r_zY=r_zY,
                    r_zZ=r_zZ,
                    nuclei_diameter=nuclei_diameter,
                    **split_cfg,
                )

                # Extract debug info for print statement
                z_anisotropy = debug_info['z_anisotropy']
                z_split_weight = debug_info['z_split_weight']
                scales_with_seeds = debug_info['scales_with_seeds']
                added_peak_seed_count = debug_info['added_peak_seed_count']
                added_seed_count = debug_info['added_seed_count']
                restored_isolated_count = debug_info['restored_isolated_count']
                erosion_triplets = debug_info['erosion_triplets']
                boundary_components = debug_info['boundary_components']
                dmin = debug_info['dmin']
                dmax = debug_info['dmax']
                n_scales = debug_info['n_scales']
                min_seed_vox = debug_info['min_seed_vox']
                removed_by_roundness = debug_info['removed_by_roundness']

                er_z_values = [e[0] for e in erosion_triplets] if erosion_triplets else [1]
                er_y_values = [e[1] for e in erosion_triplets] if erosion_triplets else [1]
                er_x_values = [e[2] for e in erosion_triplets] if erosion_triplets else [1]

                print(
                    f"Nuclei found: {int(im_out.max())} "
                    f"(shrink_factor={split_cfg['nuclei_bridge_shrink_factor']}, "
                    f"diameter_range=[{dmin:.2f},{dmax:.2f}]x, scales={n_scales}, "
                    f"scales_with_seeds={scales_with_seeds}, peak_seeds={added_peak_seed_count}, "
                    f"z_anisotropy={z_anisotropy:.2f}, z_weight={z_split_weight:.2f}, "
                    f"erosion Z={min(er_z_values)}-{max(er_z_values)} "
                    f"Y={min(er_y_values)}-{max(er_y_values)} "
                    f"X={min(er_x_values)}-{max(er_x_values)} vox, "
                    f"min_seed_vox={min_seed_vox} (boundary={3}), "
                    f"boundary_components={len(boundary_components)}, "
                    f"added_component_seeds={added_seed_count}, "
                    f"restored_isolated_voxels={restored_isolated_count}, "
                    f"removed_by_roundness={removed_by_roundness})"
                )

            im_segmentation_stack = {'Nuclei': im_out}

            cm_rand = np.random.rand(int(np.max(im_segmentation_stack['Nuclei'])), 3)
            cm_rand[0, :] = [0.0, 0.0, 0.0]

            colormaps_rand = Colormap(cm_rand)

In [ ]:
# Segmentation of cytoplasm
im_in = im_final_stack['Threshold image'].copy()

if 'im_segmentation_stack' not in globals() or 'Nuclei' not in im_segmentation_stack:
    raise RuntimeError("Run cell 38 first so nuclei segmentation creates im_segmentation_stack['Nuclei'].")

if ('NUCLEI' in stain_df.index) | ('CYTOPLASM' in stain_df.index) | len(cyto_markers) > 0:
    im_out = np.zeros_like(im_in[:, :, :, 0], dtype=np.int32)

    for c in step_progress(range(im_in.shape[3]), desc='Step 11A - Segment Cytoplasm'):
        if stain_df.index[c] == 'CYTOPLASM':
            distance = ndi.distance_transform_edt(im_in[:, :, :, c], sampling=[r_zZ, r_zY, r_zX])
            radius_X = int((nuclei_diameter / 2.0) / r_zX)
            radius_Y = int((nuclei_diameter / 2.0) / r_zY)
            radius_Z = int((nuclei_diameter / 2.0) / r_zZ)
            coords = peak_local_max(
                distance,
                footprint=make_anisotropic_footprint(radius_Z, radius_Y, radius_X),
                labels=im_in[:, :, :, c].astype(np.int32),
            )
            mask = np.zeros(distance.shape, dtype=bool)
            mask[tuple(coords.T)] = True
            markers, _ = label(mask)
            im_out = watershed(-distance, im_segmentation_stack['Nuclei'], mask=im_in[:, :, :, c])

    if 'CYTOPLASM' not in stain_df.index:
        if len(cyto_markers) == 0:
            im_out = grow_labels(im_segmentation_stack['Nuclei'], cyto_factor)
        else:
            im_out = im_segmentation_stack['Nuclei'] > 0
            for c in step_progress(range(im_in.shape[3]), desc='Step 11B - Apply Cyto Markers'):
                idx = stain_complete_df.index[c]
                marker = stain_complete_df.loc[idx, 'Marker']
                if marker in cyto_markers:
                    im_out = im_out + im_in[:, :, :, c]
            binary_mask = (im_out > 0).copy()
            im_out = grow_markers_within_islands_limited(
                im_segmentation_stack['Nuclei'], binary_mask, max_distance=10.0
            )

        stain_complete_df.loc['CYTOPLASM'] = ['', '', '', '', '', '']

    im_segmentation_stack['Cytoplasm'] = im_out.copy()

In [ ]:
# Segmentation of the pericellular matrix (PCM)
im_in=im_final_stack['Threshold image'].copy()

if ('NUCLEI' in stain_df.index)|('CYTOPLASM' in stain_df.index):
    im_out = np.zeros_like(im_in[:,:,:,0], dtype=np.int32)
    if len(cyto_markers)==0:
        im_out=grow_labels(im_segmentation_stack['Nuclei'], PCM_factor)
    else:
        P_factor = int(PCM_factor-cyto_factor)
        im_out=grow_labels(im_segmentation_stack['Cytoplasm'], P_factor)
    im_out=im_out-im_segmentation_stack['Cytoplasm']
        
    im_segmentation_stack['PCM'] = im_out.copy()

In [ ]:
# Assign segmented nuclei labels to other channels (cell assignment)
im_in=im_final_stack['Threshold image'].copy()

if ('NUCLEI' in stain_df.index)|('CYTOPLASM' in stain_df.index):
    for c in step_progress(range(im_in.shape[3]), desc='Step 12 - Assign Labels To Channels'):
        if (stain_df.index[c] != 'NUCLEI') & (stain_df.index[c] != 'CYTOPLASM') & (stain_df.index[c] != 'PCM'):
            im_segmentation_stack[stain_df.index[c]] = im_in[:, :, :, c] * (im_segmentation_stack['Cytoplasm'] + im_segmentation_stack['PCM'])
            im_segmentation_stack[stain_df.index[c] + '_cyto'] = im_in[:, :, :, c] * (im_segmentation_stack['Cytoplasm'])
            im_segmentation_stack[stain_df.index[c] + '_PCM'] = im_in[:, :, :, c] * (im_segmentation_stack['PCM'])

In [ ]:
# # Evaluate aggregates
# if ('NUCLEI' in stain_df.index)|('CYTOPLASM' in stain_df.index):
#     im_out,num_aggregates=label(grow_labels(im_segmentation_stack['Cytoplasm'],2.0)>0)
#     im_segmentation_stack['Aggregates']=im_out*(im_segmentation_stack['Cytoplasm']>0)

In [ ]:
# Visualize original, denoised, filtered, corrected, thresholded, assigned, and segmented images
viewer_0 = napari.Viewer()
scale_zoom=(r_zZ, r_zY, r_zX)

for c in step_progress(range(im_in.shape[3]), desc='Step 13A - Add Layers To Viewer 0'):
    idx = stain_complete_df.index[c]
    marker = stain_complete_df.loc[idx, 'Marker']
    color = stain_complete_df['Color'].iloc[c]
    #viewer_0.add_image(im_final_stack['Normalized image'], name=f'NORMALIZED {idx} ({marker})', colormap=color, blending='additive')
    viewer_0.add_image(im_final_stack['Original image'][:, :, :, c], name=f'ORIGINAL {idx} ({marker})', colormap=color, blending='additive', scale=[r_Z, r_Y, r_X])
    viewer_0.add_image(im_final_stack['Zoomed image'][:, :, :, c], name=f'ZOOMED {idx} ({marker})', colormap=color, blending='additive', scale=scale_zoom)
    viewer_0.add_image(im_final_stack['Denoised image'][:, :, :, c], name=f'DENOISED {idx} ({marker})', colormap=color, blending='additive', scale=scale_zoom)
    viewer_0.add_image(im_final_stack['Adjusted image'][:, :, :, c], name=f'CORRECTED {idx} ({marker})', colormap=color, blending='additive', scale=scale_zoom)
    viewer_0.add_image(im_final_stack['Filtered image'][:, :, :, c], name=f'FILTERED {idx} ({marker})', colormap=color, blending='additive', scale=scale_zoom)
    viewer_0.add_image(im_final_stack['Equalized image'][:, :, :, c], name=f'EQ {idx} ({marker})', colormap=color, blending='additive', scale=scale_zoom)
    #viewer_0.add_image(im_final_stack['Threshold image'][:, :, :, c].astype('uint8'), name=f'THRESHOLD {idx} ({marker})', contrast_limits=[0, 1], colormap=color, blending='additive', scale=scale_zoom)    
    #if stain_complete_df.index[c] == 'NUCLEI':
        # Keep full integer label IDs to avoid wrap-around when nuclei count > 255.
        #viewer_0.add_labels(im_segmentation_stack['Nuclei'].astype(np.int32), name=f'{idx} ({marker})', blending='additive', scale=scale_zoom) #, colormap=colormaps_rand, contrast_limits=[0, np.max(im_nuclei_segmented)], blending='additive')
viewer_0.scale_bar.visible = True
viewer_0.scale_bar.unit = 'um'

if ('NUCLEI' in stain_complete_df.index)|('CYTOPLASM' in stain_complete_df.index):
    viewer_1 = napari.Viewer()

    im_in=im_final_stack['Threshold image'].copy()
    
    for c in step_progress(range(len(stain_complete_df.index)), desc='Step 13B - Add Labels To Viewer 1'):
        idx = stain_complete_df.index[c]
        marker = stain_complete_df.loc[idx, 'Marker']
        if stain_complete_df.index[c] == 'NUCLEI':
            viewer_1.add_labels(im_segmentation_stack['Nuclei'].astype(np.int32), name=f'{idx} ({marker})', blending='additive', scale=scale_zoom) #, colormap=colormaps_rand, contrast_limits=[0, np.max(im_nuclei_segmented)], blending='additive')
        if stain_complete_df.index[c] == 'CYTOPLASM':
            viewer_1.add_labels(im_segmentation_stack['Cytoplasm'].astype(np.int32), blending='additive', name=f'{idx} ({marker})', scale=scale_zoom) #, colormap=colormaps_rand, contrast_limits=[0, np.max(im_nuclei_segmented)], blending='additive')
            viewer_1.add_labels(im_segmentation_stack['PCM'].astype(np.int32), name=f'PCM', blending='additive', scale=scale_zoom) #, colormap=colormaps_rand, contrast_limits=[0, np.max(im_nuclei_segmented)], blending='additive')
            #viewer_1.add_labels(im_segmentation_stack['Aggregates'].astype('uint8'), name=f'AGGREGTES', blending='additive', scale=scale_zoom) #, colormap=colormaps_rand, contrast_limits=[0, np.max(im_aggregates)], blending='additive')
        if (stain_complete_df.index[c] != 'NUCLEI') & (stain_complete_df.index[c] != 'CYTOPLASM') & (stain_complete_df.index[c] != 'PCM'):
            viewer_1.add_labels(im_segmentation_stack[stain_df.index[c]].astype(np.int32), name=f'{idx} ({marker})', blending='additive', scale=scale_zoom) #, colormap=colormaps_rand, contrast_limits=[0, np.max(im_nuclei_segmented)], blending='additive')
    viewer_1.scale_bar.visible = True
    viewer_1.scale_bar.unit = 'um'

# 6. Quantification and Analysis

This section quantifies nuclei and cell properties, computes statistics, and visualizes distributions. Results are exported for further analysis.

In [ ]:
# Quantify marker overlap and intensity per segmented object
filtered_img = im_final_stack['Filtered image']
r_xyz = (r_zX, r_zY, r_zZ)

labels_dict = build_labels_dict(
    im_segmentation_stack,
    filtered_img,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_xyz=r_xyz,
    zooms=zoom_factors,
    multilabel=multilabel,
    progress=step_progress,
)

In [ ]:
# Create DataFrames for quantification results
labels_df, truncated_df = labels_dict_to_dataframe(
    labels_dict,
    truncate=True,
    progress=step_progress,
)

In [ ]:
# Display quantification DataFrame
labels_df

In [ ]:
# Print summary statistics for nuclei and cell populations
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=step_progress,
)

In [ ]:
# Build the full quantification table used for exports and 3D outputs
r_xyz = (r_X, r_Y, r_Z)

labels_full_dict = build_full_labels_dict(
    im_segmentation_stack,
    im_final_stack,
    filtered_img,
    stain_complete_df=stain_complete_df,
    r_xyz=r_xyz,
    zooms=zoom_factors,
    progress=step_progress,
)

In [ ]:
# Create DataFrame for full quantification results
labels_full_df = labels_dict_to_dataframe(labels_full_dict)

### HISTOGRAMS

In [ ]:
# Collect per-nucleus histogram values for each non-nuclear marker
hist_data, intensity_ranges = collect_histogram_data(
    im_segmentation_stack,
    filtered_img,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    progress=step_progress,
)

seg_stack = im_segmentation_stack
seg_final = im_final_stack

In [ ]:
# Plot KDE distributions for a subset of nuclei and keep x_grid for the PDF report
fig, axes, x_grid = plot_nucleus_kdes(
    hist_data,
    stain_complete_df=stain_complete_df,
    progress=step_progress,
)

In [ ]:
# Full block: one nucleus per page, 3 channel crops + merged in one horizontal row
styles = getSampleStyleSheet()

create_row_pdf(
    output_pdf=Path(input_file).stem + "_nuclei_marker.pdf",
    pad=20,
    thumb_size=(2.0 * inch, 2.0 * inch)
)

In [ ]:
nuc_3D_export=False

# --- VTK 3D crop of a single nucleus ----------------------------------------
# Saves a 30x30x30 voxel cube centered on the chosen nucleus as a VTK file.
# Each voxel stores:
#   Nuclei_label     : value from im_segmentation_stack['Nuclei']
#   Cytoplasm_label  : value from im_segmentation_stack['Cytoplasm']
#   <COND>_<MARKER>  : equalized intensity from im_final_stack['Equalized image']
#                      for every non-NUCLEI channel
# -----------------------------------------------------------------------------
if nuc_3D_export:
    nuc_label = int(input("Enter nucleus label number to export: "))
    
    # -- centroid -----------------------------------------------------------------
    nuc_vol = im_segmentation_stack['Nuclei']           # shape (Z, Y, X)
    mask = (nuc_vol == nuc_label)
    if not np.any(mask):
        raise ValueError(f"Nucleus {nuc_label} not found in im_segmentation_stack['Nuclei'].")
    
    if 'Cytoplasm' not in im_segmentation_stack:
        raise ValueError("im_segmentation_stack['Cytoplasm'] is missing. Run cytoplasm segmentation first.")
    
    cyto_vol = im_segmentation_stack['Cytoplasm']
    
    coords = np.argwhere(mask)                          # (N, 3): each row = [z, y, x]
    centroid = np.round(coords.mean(axis=0)).astype(int)
    cz, cy, cx = centroid
    print(f"Nucleus {nuc_label} - centroid  Z={cz}  Y={cy}  X={cx}")
    
    # -- 30x30x30 bounding box ----------------------------------------------------
    SIZE = 90
    half = SIZE // 2
    Zmax, Ymax, Xmax = nuc_vol.shape
    
    z0d, z1d = cz - half, cz + half
    y0d, y1d = cy - half, cy + half
    x0d, x1d = cx - half, cx + half
    
    # clamp to image boundaries
    z0c, z1c = max(0, z0d), min(Zmax, z1d)
    y0c, y1c = max(0, y0d), min(Ymax, y1d)
    x0c, x1c = max(0, x0d), min(Xmax, x1d)
    
    # write extents inside the padded output cube
    zp0, zp1 = z0c - z0d, z0c - z0d + (z1c - z0c)
    yp0, yp1 = y0c - y0d, y0c - y0d + (y1c - y0c)
    xp0, xp1 = x0c - x0d, x0c - x0d + (x1c - x0c)
    
    def _crop3d(vol):
        """Zero-padded SIZE^3 sub-volume extracted from a (Z, Y, X) array."""
        out = np.zeros((SIZE, SIZE, SIZE), dtype=vol.dtype)
        out[zp0:zp1, yp0:yp1, xp0:xp1] = vol[z0c:z1c, y0c:y1c, x0c:x1c]
        return out
    
    # -- build per-channel arrays -------------------------------------------------
    nuclei_crop = _crop3d(nuc_vol).astype(np.int32)
    cytoplasm_crop = _crop3d(cyto_vol).astype(np.int32)
    
    eq_img = im_final_stack['Equalized image']          # shape (Z, Y, X, C)
    marker_crops = {}
    for c_idx in range(eq_img.shape[3]):
        cond = stain_df.index[c_idx]
        if cond != 'NUCLEI':
            marker = stain_df['Marker'].iloc[c_idx]
            ch_name = f"{cond}_{marker}".replace(" ", "_").replace("-", "_")
            marker_crops[ch_name] = _crop3d(eq_img[:, :, :, c_idx]).astype(np.float32)
    
    # -- assemble pyvista ImageData -----------------------------------------------
    # DIMENSIONS = (nx, ny, nz) points; origin = (x0, y0, z0) in VTK (x,y,z) order.
    # VTK point ordering: x (i) varies fastest -> numpy arr[Z,Y,X].ravel() is correct.
    grid = pv.ImageData()
    grid.dimensions = (SIZE, SIZE, SIZE)
    grid.origin = (float(x0d), float(y0d), float(z0d))
    grid.spacing = (1.0, 1.0, 1.0)
    
    grid.point_data['Nuclei_label'] = nuclei_crop.ravel()
    grid.point_data['Cytoplasm_label'] = cytoplasm_crop.ravel()
    for ch_name, crop in marker_crops.items():
        grid.point_data[ch_name] = crop.ravel()
    
    # -- save ---------------------------------------------------------------------
    out_path = str(Path(input_file).stem + f"_nuc{nuc_label}_3Dcrop.vtk")
    grid.save(out_path)
    print(f"Saved: {out_path}  ({SIZE}^3 voxels, {2 + len(marker_crops)} channels)")

## Evaluate cell distribution in the space

In [ ]:
# Plot spatial distribution of nuclei and cells
im_in = im_final_stack['Filtered image']
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_in=im_in,
    r_X=r_X,
    r_Y=r_Y,
    r_Z=r_Z,
    zoom_factors=zoom_factors,
    progress=step_progress,
)

## Evaluate cell size distribution

In [ ]:
# Plot size distribution of nuclei and cells
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=step_progress,
)

## CREATE .VTK VOLUME

In [ ]:
diamond = ndi.generate_binary_structure(rank=3, connectivity=1)
blocks_nuclei=pv.MultiBlock()
blocks_cyto=pv.MultiBlock()
blocks_PCM=pv.MultiBlock()
nuclei_stl_old=mr.Mesh()
cyto_stl_old=mr.Mesh()
PCM_stl_old=mr.Mesh()

nuc_vol=np.zeros((np.max(im_segmentation_stack['Nuclei'])+1,))
nuc_coord=np.zeros((np.max(im_segmentation_stack['Nuclei'])+1,3))
nuc_list=np.zeros((np.max(im_segmentation_stack['Nuclei'])+1,))

cyto_vol=np.zeros((np.max(im_segmentation_stack['Cytoplasm'])+1,))
cyto_coord=np.zeros((np.max(im_segmentation_stack['Cytoplasm'])+1,3))
cyto_list=np.zeros((np.max(im_segmentation_stack['Cytoplasm'])+1,))

PCM_vol=np.zeros((np.max(im_segmentation_stack['PCM'])+1,))
PCM_coord=np.zeros((np.max(im_segmentation_stack['PCM'])+1,3))
PCM_list=np.zeros((np.max(im_segmentation_stack['PCM'])+1,))

#agg_id=1

k=0
for j in step_progress(range(1,np.max(im_segmentation_stack['Nuclei'])+1), desc='Step 18 - Build VTK Volumes'):
    clear_output(wait=True)
    print('NUCLEI ' + str(j) + ' / ' + str(np.max(im_segmentation_stack['Nuclei'])))
    
    simpleVolume = mrn.simpleVolumeFrom3Darray(np.float32(im_segmentation_stack['Nuclei']==j))
    floatGrid = mr.simpleVolumeToDenseGrid(simpleVolume)
    mesh_stl = mr.gridToMesh(floatGrid , mr.Vector3f(1.0,1.0,1.0), 0.5)    
    mr.saveMesh(mesh_stl, "part_nuclei_mesh.stl" )
    
    mesh_nuclei = pv.read("part_nuclei_mesh.stl")
    if mesh_nuclei.volume>0.0:
        mesh_nuclei.decimate(target_reduction=0.8, inplace=True)

        nuc_vol[k]=mesh_nuclei.volume
        nuc_coord[k]=mesh_nuclei.center
        nuc_list[k]=j

        mesh_nuclei.cell_data['ID']=np.ones(mesh_nuclei.n_cells)*(k+1)
        mesh_nuclei.cell_data['Nuclei volume (um3)']=np.ones(mesh_nuclei.n_cells)*nuc_vol[k] * r_X * r_Y * r_Z / np.prod(zoom_factors)
        mesh_nuclei.cell_data['Z nuclei (um)']=np.ones(mesh_nuclei.n_cells)*nuc_coord[k][0] * r_Z /zoom_factors[0]
        mesh_nuclei.cell_data['Y nuclei (um)']=np.ones(mesh_nuclei.n_cells)*nuc_coord[k][1] * r_Y /zoom_factors[1]
        mesh_nuclei.cell_data['X nuclei (um)']=np.ones(mesh_nuclei.n_cells)*nuc_coord[k][2] * r_X /zoom_factors[2]
        
        blocks_nuclei.append(mesh_nuclei)
        k=k+1


    simpleVolume = mrn.simpleVolumeFrom3Darray(np.float32(im_segmentation_stack['Cytoplasm']==j))
    floatGrid = mr.simpleVolumeToDenseGrid(simpleVolume)
    mesh_stl = mr.gridToMesh(floatGrid , mr.Vector3f(1.0,1.0,1.0), 0.5)    
    mr.saveMesh(mesh_stl, "part_cyto_mesh.stl" )
    
    mesh_cyto = pv.read("part_cyto_mesh.stl")

    simpleVolume = mrn.simpleVolumeFrom3Darray(np.float32(im_segmentation_stack['PCM']==j))
    floatGrid = mr.simpleVolumeToDenseGrid(simpleVolume)
    mesh_stl = mr.gridToMesh(floatGrid , mr.Vector3f(1.0,1.0,1.0), 0.5)    
    mr.saveMesh(mesh_stl, "part_PCM_mesh.stl" )
    
    mesh_PCM = pv.read("part_PCM_mesh.stl")
    
    if mesh_cyto.volume>0.0:
        mesh_cyto.decimate(target_reduction=0.8, inplace=True)
        mesh_PCM.decimate(target_reduction=0.8, inplace=True)

        cyto_vol[k]=mesh_cyto.volume
        cyto_coord[k]=mesh_cyto.center
        cyto_list[k]=j

        PCM_vol[k]=mesh_PCM.volume
        PCM_coord[k]=mesh_PCM.center
        PCM_list[k]=j

        mesh_cyto.cell_data['ID']=np.ones(mesh_cyto.n_cells)*(k+1)
        mesh_PCM.cell_data['ID']=np.ones(mesh_PCM.n_cells)*(k+1)
        mesh_cyto.cell_data['Cellular volume (um3)']=np.ones(mesh_cyto.n_cells)*cyto_vol[k] * r_X * r_Y * r_Z / np.prod(zoom_factors)
        mesh_PCM.cell_data['PCM volume (um3)']=np.ones(mesh_PCM.n_cells)*PCM_vol[k] * r_X * r_Y * r_Z / np.prod(zoom_factors)
        mesh_cyto.cell_data['Z cell (um)']=np.ones(mesh_cyto.n_cells)*cyto_coord[k][0] * r_Z /zoom_factors[0]
        mesh_cyto.cell_data['Y cell (um)']=np.ones(mesh_cyto.n_cells)*cyto_coord[k][1] * r_Y /zoom_factors[1]
        mesh_cyto.cell_data['X cell (um)']=np.ones(mesh_cyto.n_cells)*cyto_coord[k][2] * r_X /zoom_factors[2]
        mesh_PCM.cell_data['Z PCM (um)']=np.ones(mesh_PCM.n_cells)*PCM_coord[k][0] * r_Z /zoom_factors[0]
        mesh_PCM.cell_data['Y PCM (um)']=np.ones(mesh_PCM.n_cells)*PCM_coord[k][1] * r_Y /zoom_factors[1]
        mesh_PCM.cell_data['X PCM (um)']=np.ones(mesh_PCM.n_cells)*PCM_coord[k][2] * r_X /zoom_factors[2]
        for i, marker in enumerate(labels_full_df.index):
            if (labels_full_df['Condition'][i]!='NUCLEI') & (labels_full_df['Condition'][i]!='CYTOPLASM') & (np.size(marker)==1):
                if j in list(labels_full_df['Shared labels'][i]):
                    mesh_cyto.cell_data[marker+' volume (um3)']=np.ones(mesh_cyto.n_cells)*(labels_full_df['Marker size [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_PCM.cell_data[marker+' volume (um3)']=np.ones(mesh_PCM.n_cells)*(labels_full_df['Marker size [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_cyto.cell_data[marker+' volume cytoplasm (um3)']=np.ones(mesh_cyto.n_cells)*(labels_full_df['Marker size cytoplasm [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_PCM.cell_data[marker+' volume PCM (um3)']=np.ones(mesh_PCM.n_cells)*(labels_full_df['Marker size PCM [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_cyto.cell_data[marker+' rel. vol. (-)']=np.ones(mesh_cyto.n_cells)*((labels_full_df['Marker size [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])/((cyto_vol[k]+PCM_vol[k]) * r_X * r_Y * r_Z / np.prod(zoom_factors)))
                    mesh_PCM.cell_data[marker+' rel. vol. (-)']=np.ones(mesh_PCM.n_cells)*((labels_full_df['Marker size [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])/((cyto_vol[k]+PCM_vol[k]) * r_X * r_Y * r_Z / np.prod(zoom_factors)))
                    mesh_cyto.cell_data[marker+' rel. vol. cytoplasm (-)']=np.ones(mesh_cyto.n_cells)*((labels_full_df['Marker size cytoplasm [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])/(cyto_vol[k] * r_X * r_Y * r_Z / np.prod(zoom_factors)))
                    mesh_PCM.cell_data[marker+' rel. vol. PCM (-)']=np.ones(mesh_PCM.n_cells)*((labels_full_df['Marker size PCM [um3]'][i][list(labels_full_df['Shared labels'][i]).index(j)])/(PCM_vol[k] * r_X * r_Y * r_Z / np.prod(zoom_factors)))
                    mesh_cyto.cell_data[marker+' avg. intensity (-)']=np.ones(mesh_cyto.n_cells)*(labels_full_df['Avg. marker intensity'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_PCM.cell_data[marker+' avg. intensity (-)']=np.ones(mesh_PCM.n_cells)*(labels_full_df['Avg. marker intensity'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_cyto.cell_data[marker+' avg. cytoplasm int. (-)']=np.ones(mesh_cyto.n_cells)*(labels_full_df['Avg. marker intensity cytoplasm'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                    mesh_PCM.cell_data[marker+' avg. PCM int. (-)']=np.ones(mesh_PCM.n_cells)*(labels_full_df['Avg. marker intensity PCM'][i][list(labels_full_df['Shared labels'][i]).index(j)])
                else:
                    mesh_cyto.cell_data[marker+' expression (um3)']=np.ones(mesh_cyto.n_cells)*(0.0)
                    mesh_cyto.cell_data[marker+' rel. expr. (-)']=np.ones(mesh_cyto.n_cells)*(0.0)
                # ass_channel_2=globals()[channel+'mag']*(NUCLEIlab==val)/np.max(globals()[channel+'mag'])
                # mesh_cyto.cell_data[channel+'_perc_rel']=np.ones(mesh_nuclei.n_cells)*(np.sum(ass_channel_2)/np.sum(NUCLEIlab==val))
        
        blocks_cyto.append(mesh_cyto)
        blocks_PCM.append(mesh_PCM)
        #k=k+1

    #j=j-1

# nuc_vol=nuc_vol[0:k-1]
# nuc_coord=nuc_coord[0:k-1]
# nuc_list=nuc_list[0:k-1]
blocks_nuclei.extract_geometry().save(Path(input_file).stem+'_NUCLEI_labelled.vtk')
blocks_cyto.extract_geometry().save(Path(input_file).stem+'_CYTOPLASM_labelled.vtk')
blocks_PCM.extract_geometry().save(Path(input_file).stem+'_PCM_labelled.vtk')

## and .STL for markers

In [ ]:
for c, marker in step_progress(enumerate(stain_complete_df.index), total=len(stain_complete_df.index), desc='Step 19 - Export Marker STL'):
    if (stain_complete_df.index[c] != 'NUCLEI') & (stain_complete_df.index[c] != 'CYTOPLASM') & (stain_complete_df.index[c] != 'PCM'):
        simpleVolume = mrn.simpleVolumeFrom3Darray(np.float32(im_segmentation_stack[stain_df.index[c]]>0))
        floatGrid = mr.simpleVolumeToDenseGrid(simpleVolume)
        mesh_stl = mr.gridToMesh(floatGrid , mr.Vector3f(1.0,1.0,1.0), 0.5)    
        mr.saveMesh(mesh_stl,Path(input_file).stem + "_" + stain_complete_df['Marker'][c] + "_mesh.stl" )

### Create a complete report XSL

In [ ]:
# Export quantification results to Excel file
output_path = Path(input_file).stem + '_segmentation.xlsx'
export_quantification_to_excel(
    output_path,
    original_stain_complete_df,
    labels_full_df,
    progress=step_progress,
)
print(f"Saved to: {output_path}")

# CREATE .inp FOR FINITE ELEMENT ANALYSIS

In [ ]:
simpleVolume = mrn.simpleVolumeFrom3Darray(np.float32(im_segmentation_stack['Nuclei']))
floatGrid = mr.simpleVolumeToDenseGrid(simpleVolume)
mesh_stl = mr.gridToMesh(floatGrid , mr.Vector3f(1.0,1.0,1.0), 0.5)

outVerts = mrn.getNumpyVerts(mesh_stl)
#print(outVerts)

outFaces = mrn.getNumpyFaces(mesh_stl.topology)

tet = tetgen.TetGen(outVerts,outFaces)
nodes,elems=tet.tetrahedralize(order=1, mindihedral=20, minratio=1.5)

tet.write('FE_segmentation_full.vtk', binary=False)

In [ ]:
meshel = meshio.read('FE_segmentation_full.vtk')
meshel.write('FE_segmentation.inp')

for c in step_progress(range(1, np.max(im_segmentation_stack['Nuclei'])+1), desc='Step 21A - Initialize Element Sets'):
    globals()[str(c)+'cell_el']=[]

for ce, x in step_progress(enumerate(elems), total=len(elems), desc='Step 21B - Assign Elements To Cells'):
    #print(np.shape(np.uint16(np.mean(nodes[x],0))))
    coord=np.int16(np.round(np.mean(nodes[x],0),0))
    step=0
    taken=False
    while not(taken):
        step+=1
        coord[coord<step]=1
        for k in [0,1,2]:
            if coord[k]>=np.shape(im_segmentation_stack['Nuclei'])[k]+1-step:coord[k]=np.shape(im_segmentation_stack['Nuclei'])[k]-1
        elemlist=im_segmentation_stack['Nuclei'][coord[0]-step:coord[0]+1+step,coord[1]-step:coord[1]+1+step,coord[2]-step:coord[2]+1+step].flatten()
        #print(elemlist)
        if sum(elemlist)>0:
            c_el=st.mode(elemlist[elemlist!=0])
            taken=True

    #print(c_el)
    if c_el!=0:
        globals()[str(c_el)+'cell_el'].append(ce+1)

f = open("FE_segmentation.inp", "a")
for c in step_progress(range(1,np.max(im_segmentation_stack['Nuclei'])+1), desc='Step 21C - Write Element Sets'):
    f.write("*Elset, elset=cell" + str(c) + "\n")
    j=1
    for t in range(1, np.size(globals()[str(c)+'cell_el'])):
        f.write(str(globals()[str(c)+'cell_el'][t]) + ",")
        j+=1
        if j>16:
            f.write("\n")
            j=1
    f.write("\n")

    
f.close()

In [ ]:
# Now insert *PART header manually
with open("FE_segmentation.inp", "r") as f:
    lines = f.readlines()

with open(Path(input_file).stem + "_FEA.inp", "w") as f:
    for line in step_progress(lines, desc='Step 21D - Write Final INP'):
        if (line=="*NODE\n"):
            f.write("*PART, name=Part-1\n")
        f.write(line)
    f.write("*END PART\n")